# Lab 04: File formats and data layout

**Time:** about 2 h 30 min.

**Checklist questions this lab gives you evidence for:**<br><span style="color:#d93025"><b>2.1 Why is CSV heavy?</b></span><br><span style="color:#d93025"><b>2.2 Columnar storage</b></span><br><span style="color:#d93025"><b>2.3 Schema evolution</b></span><br><span style="color:#d93025"><b>2.4 Predicate pushdown</b></span><br><span style="color:#d93025"><b>2.5 Small file problem</b></span><br><span style="color:#d93025"><b>2.6 Avro for Kafka</b></span><br><span style="color:#d93025"><b>2.7 Disadvantages of Parquet</b></span><br><span style="color:#d93025"><b>2.8 Partitioned folders</b></span><br><span style="color:#d93025"><b>2.9 Splittable files</b></span><br><span style="color:#d93025"><b>2.10 Parquet vs ORC</b></span>

**Goal:** write the same data in five formats, prove pushdown and pruning in the plan, and see what partition folders, small files and gzip do. The last section is date and complex-type practice.

## Data information
A small online shop with **one table**. The setup cell below creates it in `data/orders.parquet` (1,000,000 orders from January to March 2024; seeded, so everyone gets the same rows):
* *order_id*; *customer_id* (empty for guest checkout), *customer_name*, *country*, *segment* (`retail`, `business` or `guest`)
* *product_id*, *category* (8 categories), *price* (price of one item)
* *quantity* (1 to 5) and *amount* = quantity × price, in dollars
* *status*: `paid`, `shipped`, `cancelled` or `returned`; *payment_method*: `card`, `cash` or `wallet`
* *order_ts* and *order_date*: when the order was placed; *tracking_code*: a 32-character code

Customer 1 is "Big Reseller Ltd": a business customer who orders in bulk (quantity 5) and places 30% of all orders. Some labs make a small lookup table from this one table (for example one row per customer); the notebook says when.

## How to work
* Write your answer in the code cell under each question, then run the check cell below it: ✅ correct, ❌ try again.
* When a question says *guess first*, write your guess as a comment before you run anything.
* Keep the Spark UI open (the setup cell prints its address).
* Stuck for more than 10 minutes? Read the documentation list at the top, then the reference solution.


## Documentation
Everything you need is on these pages. The questions do not repeat them: read, try things out in a cell, and find the answers yourself.

* [Generic load/save functions](https://spark.apache.org/docs/latest/sql-data-sources-load-save-functions.html)
* [Parquet files](https://spark.apache.org/docs/latest/sql-data-sources-parquet.html)
* [ORC files](https://spark.apache.org/docs/latest/sql-data-sources-orc.html)
* [Avro data source](https://spark.apache.org/docs/latest/sql-data-sources-avro.html)
* [Parquet configuration (filter pushdown)](https://spark.apache.org/docs/latest/sql-data-sources-parquet.html#configuration)
* [Parquet file format](https://parquet.apache.org/docs/file-format/)
* [pyarrow ParquetFile](https://arrow.apache.org/docs/python/generated/pyarrow.parquet.ParquetFile.html)
* [Partition discovery](https://spark.apache.org/docs/latest/sql-data-sources-parquet.html#partition-discovery)
* [Performance tuning: partitions](https://spark.apache.org/docs/latest/sql-performance-tuning.html#tuning-partitions)
* [DataFrame.repartition](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.repartition.html)
* [CSV options (compression)](https://spark.apache.org/docs/latest/sql-data-sources-csv.html)
* [Schema merging](https://spark.apache.org/docs/latest/sql-data-sources-parquet.html#schema-merging)
* [functions.date_trunc](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.date_trunc.html)
* [functions.sequence](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.sequence.html)
* [functions.explode](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.explode.html)
* [functions.collect_set](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.collect_set.html)
* [functions.array_sort](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.array_sort.html)
* [Chapter 2: A Tour of PySpark Data Types](https://spark.apache.org/docs/latest/api/python/user_guide/touroftypes.html)
* [functions.from_json](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.from_json.html)

## Setup
Run the cell below once (its code is hidden: click it to expand). It creates the data and starts Spark.

In [ ]:
# Setup: run this cell once. It creates the shop data in data/ and starts Spark.
# Nothing to solve here, and you do not need to read it.
import contextlib, hashlib, io, json, os, re, shutil, sys, time, urllib.request
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # Spark's Python workers use this kernel's Python
from pyspark.sql import SparkSession, Window, functions as F

# ---------------------------------------------------------------- the shop data ---------
# A small online shop with ONE table, generated from seeded hash functions: every machine gets the same rows.
COUNTRIES = ["Uzbekistan", "Germany", "USA", "India", "Brazil", "Japan"]
CATEGORIES = ["books", "toys", "garden", "sports", "kitchen", "beauty", "music", "office"]
N_CUSTOMERS, N_PRODUCTS, N_ORDERS = 10_000, 200, 1_000_000

def _u(col, salt):
    """Pseudo-random number in [0, 1) computed from a column (same result on every machine)."""
    return F.pmod(F.xxhash64(col, F.lit(salt)), F.lit(1_000_000)) / 1_000_000.0

def _pick(u, options):
    return F.element_at(F.array(*[F.lit(o) for o in options]), (F.floor(u * len(options)) + 1).cast("int"))

def make_orders(spark, n=N_ORDERS, dirty=False):
    i = F.col("id")
    cust = (F.when(_u(i, 11) < 0.02, F.lit(None).cast("long"))                        # guest checkout: no customer
             .when(_u(i, 10) < 0.30, F.lit(1).cast("long"))                           # the big reseller: 30% of orders
             .otherwise((F.floor(_u(i, 12) * (N_CUSTOMERS - 1)) + 2).cast("long")))
    name = F.when(cust == 1, "Big Reseller Ltd").when(cust.isNotNull(), F.concat(F.lit("Customer "), cust.cast("string")))
    country = _pick(F.when(cust.isNotNull(), _u(cust, 1)).otherwise(_u(i, 31)), COUNTRIES)
    segment = F.when(cust.isNull(), "guest").when((cust == 1) | (_u(cust, 3) < 0.2), "business").otherwise("retail")
    product = (F.floor(_u(i, 14) * N_PRODUCTS) + 1).cast("int")
    category = F.element_at(F.array(*[F.lit(c) for c in CATEGORIES]), (product % 8 + 1).cast("int"))
    price = F.round((product * 37) % 195 + 5.99, 2)
    quantity = F.when(cust == 1, F.lit(5)).otherwise(F.floor(_u(i, 13) * 5) + 1).cast("int")   # the reseller always orders in bulk
    ts = F.lit("2024-01-01 00:00:00").cast("timestamp") + F.make_interval(secs=F.floor(_u(i, 15) * 91 * 86400).cast("int"))
    u = _u(i, 17)
    status = F.when(u < 0.55, "paid").when(u < 0.80, "shipped").when(u < 0.92, "cancelled").otherwise("returned")
    v = _u(i, 18)
    payment = F.when(v < 0.60, "card").when(v < 0.85, "cash").otherwise("wallet")
    df = spark.range(0, n, 1, 4).select(
        (i + 1).alias("order_id"), cust.alias("customer_id"), name.alias("customer_name"), country.alias("country"), segment.alias("segment"),
        product.alias("product_id"), category.alias("category"), price.alias("price"), quantity.alias("quantity"),
        F.round(quantity * price, 2).alias("amount"), status.alias("status"), payment.alias("payment_method"),
        ts.alias("order_ts"), F.md5(i.cast("string")).alias("tracking_code"))
    df = df.withColumn("order_date", F.to_date("order_ts"))
    if dirty:    # the monthly files of the final project contain mistakes on purpose
        df = df.withColumn("amount", F.when(_u(F.col("order_id"), 19) < 0.008, -F.col("amount")).otherwise(F.col("amount")))
        df = df.withColumn("quantity", F.when(_u(F.col("order_id"), 20) < 0.006, 0).otherwise(F.col("quantity")))
        df = df.unionByName(df.filter(_u(F.col("order_id"), 21) < 0.005))              # exact duplicates
    return df.select("order_id", "customer_id", "customer_name", "country", "segment", "product_id", "category", "price", "quantity",
                     "amount", "status", "payment_method", "order_ts", "order_date", "tracking_code")

def _create_data(spark):
    os.makedirs("data/monthly", exist_ok=True)
    make_orders(spark).repartition(4).write.mode("overwrite").parquet("data/orders.parquet")
    dirty = make_orders(spark, dirty=True)
    wrong = _u(F.col("order_id"), 22) < 0.003          # a few orders land in the wrong month's file
    month = F.month("order_date")
    file_month = F.when(wrong, month % 3 + 1).otherwise(month)
    for m in (1, 2, 3):
        (dirty.filter(file_month == m).coalesce(1).write.mode("overwrite")
              .parquet(f"data/monthly/orders_2024-{m:02d}.parquet"))

# ---------------------------------------------------------------- the Spark session ----
def new_session(master="local[4]", driver_memory="2g", delta=False, avro=False, aqe=False, conf=None):
    builder = (SparkSession.builder.master(master).appName("day04_lab")
               .config("spark.driver.memory", driver_memory)
               .config("spark.ui.showConsoleProgress", "false")
               .config("spark.sql.adaptive.enabled", str(aqe).lower())   # off = exact job and stage counts
               .config("spark.sql.session.timeZone", "UTC")                # same dates and hours for everyone
               .config("spark.sql.warehouse.dir", os.path.abspath("spark-warehouse"))
               # Spark 4.2.0 bug (SPARK-60294): Python UDFs on Parquet can hang without this.
               .config("spark.sql.execution.arrow.pythonUDF.columnarInput.enabled", "false"))
    for k, v in (conf or {}).items():
        builder = builder.config(k, v)
    if delta:
        from delta import configure_spark_with_delta_pip
        builder = (builder.config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
                          .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
        builder = configure_spark_with_delta_pip(builder, extra_packages=["org.apache.spark:spark-avro_2.13:4.2.0"] if avro else None)
    elif avro:
        builder = builder.config("spark.jars.packages", "org.apache.spark:spark-avro_2.13:4.2.0")
    s = builder.getOrCreate()
    s.sparkContext.setLogLevel("ERROR")
    return s

for _d in ("spark-warehouse", "metastore_db"):
    shutil.rmtree(_d, ignore_errors=True)               # a clean catalog on every start
spark = new_session(avro=True)
def _data_ok():
    """True if the single-table shop data is already there (an older three-table version is rebuilt)."""
    try:
        return "customer_name" in spark.read.parquet("data/orders.parquet").columns and os.path.exists("data/monthly/orders_2024-03.parquet")
    except Exception:
        return False

if not _data_ok():
    print("creating the shop data (about 30 seconds, once)...")
    _create_data(spark)
print(f"Spark {spark.version} | UI: {spark.sparkContext.uiWebUrl}")

# ---------------------------------------------------------------- helpers ----------------
def _tracker():
    return spark.sparkContext.statusTracker()

def _run_group(action):
    sc = spark.sparkContext
    group = f"g{time.time_ns()}"
    sc.setJobGroup(group, group)
    try:
        action()
    finally:
        sc.setLocalProperty("spark.jobGroup.id", None)
    return [_tracker().getJobInfo(j) for j in sorted(_tracker().getJobIdsForGroup(group))]

def jobs_run(action):
    """How many Spark jobs calling action() starts."""
    return len(_run_group(action))

def stages_run(action):
    """How many stages those jobs had in total."""
    return sum(len(j.stageIds) for j in _run_group(action))

def last_stage_tasks(action):
    """Number of tasks in the last stage action() ran."""
    jobs = _run_group(action)
    return _tracker().getStageInfo(max(s for j in jobs for s in j.stageIds)).numTasks

def _api(path):
    sc = spark.sparkContext
    return json.load(urllib.request.urlopen(f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}{path}"))

api = _api   # read anything from the Spark UI's REST API, e.g. api("/executors")

def _stage_rows(action):
    jobs = _run_group(action)
    ids = {s for j in jobs for s in j.stageIds}
    return [s for s in _api("/stages?length=2000") if s["stageId"] in ids]

def shuffle_bytes(action):
    """Total shuffle bytes written by the stages action() ran."""
    return sum(s["shuffleWriteBytes"] for s in _stage_rows(action))

def spill_bytes(action):
    """(memory, disk) bytes spilled by the stages action() ran."""
    rows = _stage_rows(action)
    return sum(s["memoryBytesSpilled"] for s in rows), sum(s["diskBytesSpilled"] for s in rows)

def skew_ratio(action):
    """Slowest task / median task, in the stage with the most tasks that action() ran."""
    rows = _stage_rows(action)
    st = max(rows, key=lambda s: s["numTasks"])
    tasks = _api(f"/stages/{st['stageId']}/{st['attemptId']}/taskList?length=5000")
    times = sorted(t["taskMetrics"]["executorRunTime"] for t in tasks if t.get("taskMetrics"))
    return round(times[-1] / max(times[len(times) // 2], 1), 1)

def failed_attempts():
    """Failed task attempts in the most recent failed stage (from the Spark UI's REST API)."""
    failed = _api("/stages?status=failed")
    if not failed:
        return 0
    st = max(failed, key=lambda s: s["stageId"])
    tasks = _api(f"/stages/{st['stageId']}/{st['attemptId']}/taskList?length=500")
    return sum(1 for t in tasks if t["status"] == "FAILED")

def sql_metric(action, metric):
    """Value of one scan metric (for example 'number of files read') of the query action() runs."""
    desc = f"q{time.time_ns()}"
    spark.sparkContext.setJobDescription(desc)
    try:
        action()
    finally:
        spark.sparkContext.setJobDescription(None)
    for ex in _api("/sql?details=true&planDescription=false&length=2000"):
        if ex.get("description") == desc:
            vals = [m["value"] for n in ex["nodes"] for m in n["metrics"] if m["name"] == metric]
            return vals[0] if vals else None

def noop(df):
    """Run the whole query but throw the result away."""
    df.write.format("noop").mode("overwrite").save()

def plan(df, mode="formatted"):
    """explain() output as a string."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain(mode)
    return buf.getvalue()

def count_nodes(df, *names):
    """How often each operator name appears in the simple plan."""
    text = plan(df, "simple")
    return {n: len(re.findall(r"\b" + re.escape(n) + r"\b", text)) for n in names}

def optimized(df):
    """The optimized logical plan as a string."""
    return df._jdf.queryExecution().optimizedPlan().toString()

def operators(df):
    """Operator names of the physical plan tree, top to bottom."""
    tree = plan(df).split("\n\n")[0].splitlines()[1:]
    return [re.sub(r"\s*\(\d+\)$", "", line).strip(" +-*:") for line in tree]

def rules_fired(df):
    """Names of the Catalyst rules that changed this query's plan."""
    qe = df._jdf.queryExecution(); qe.executedPlan()
    it, fired = qe.tracker().rules().iterator(), set()
    while it.hasNext():
        kv = it.next()
        if kv._2().numEffectiveInvocations() > 0:
            fired.add(kv._1().split(".")[-1].split("$")[-1])
    return fired

def estimate(df):
    """(sizeInBytes, rowCount or None) the optimizer estimates for df."""
    stats = df._jdf.queryExecution().optimizedPlan().stats()
    return int(str(stats.sizeInBytes())), (int(str(stats.rowCount().get())) if stats.rowCount().isDefined() else None)

def partition_sizes(df):
    """Rows per partition, largest first."""
    return sorted((r["count"] for r in df.groupBy(F.spark_partition_id().alias("p")).count().collect()), reverse=True)

def timeit(action):
    start = time.perf_counter(); action(); return round(time.perf_counter() - start, 2)

def folder_mb(path):
    files = [os.path.join(d, f) for d, _, fs in os.walk(path) for f in fs] if os.path.isdir(path) else [path]
    return round(sum(os.path.getsize(f) for f in files) / 1e6, 1)

# ---------------------------------------------------------------- checks -----------------
_EXPECTED = {
 "4.2": "eedf2a4831f29330",
 "4.4": "1da51b8d8ff98f6a",
 "4.6": "364b1cb2eec39fb2",
 "4.7": "4755e0e37795e8be",
 "4.8": "7afc03a982adf3fa",
 "4.9": "53a583621eae90b7",
 "4.10": "e030acacebab8cd9",
 "4.11": "3241eb849ccf2e48",
 "4.12": "a2ae5d34f38b3056"
}
_RECORD = os.environ.get("LAB_RECORD")

def _norm(v):
    if isinstance(v, bool) or v is None: return v
    if isinstance(v, int): return v
    if isinstance(v, float): return round(v, 2)
    if isinstance(v, str): return v.strip()
    if isinstance(v, dict): return {str(k): _norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
    if isinstance(v, (list, tuple)): return [_norm(x) for x in v]
    if isinstance(v, (set, frozenset)): return sorted((_norm(x) for x in v), key=json.dumps)
    if hasattr(v, "item"): return _norm(v.item())
    if type(v).__name__ == "Decimal": return round(float(v), 2)
    raise TypeError(f"check() needs plain Python values, got {type(v).__name__}")

def check(task, value):
    normal = _norm(value)
    if _RECORD:
        rec = json.load(open(_RECORD)) if os.path.exists(_RECORD) else {}
        rec[task] = normal; json.dump(rec, open(_RECORD, "w"))
    ok = hashlib.sha256(json.dumps(normal).encode()).hexdigest()[:16] == _EXPECTED.get(task)
    print(f"✅ {task} correct" if ok else f"❌ {task} not yet: check your code and the question again")

def check_true(task, condition, hint=""):
    print(f"✅ {task} correct" if condition else f"❌ {task} not yet. {hint}")


Load the data the way Lab 01 taught you (given):

In [ ]:
orders = spark.read.parquet("data/orders.parquet")

## A. Five formats, one dataset

**4.1 Write *orders* as CSV (with header), JSON, Parquet, ORC and Avro into `formats/<format>`. Which is the biggest, and which is the fastest to read?**
* *sizes* = dict `{format: folder size in MB}` (use `folder_mb(path)`)
* *read_times* = dict `{format: seconds}` for reading it back (CSV and JSON with `orders.schema`) and computing `sum(amount)`, timed with `timeit`

In [ ]:
check_true("4.1", sizes["json"] == max(sizes.values()) and sizes["parquet"] < sizes["csv"] and sizes["orc"] < sizes["csv"] and read_times["parquet"] < read_times["csv"],
           "JSON should be the biggest, Parquet and ORC smaller than CSV, and Parquet faster to read than CSV")

**4.2 What does Spark push down and read for `parquet_df.filter(amount > 900).select("order_id", "amount")`?** Load the Parquet copy as *parquet_df*. Copy the *PushedFilters* from the plan into *pushed* (list of strings) and the columns in *ReadSchema* into *read_cols* (sorted names).

In [ ]:
check("4.2", [sorted(pushed), read_cols])

**4.3 Why does sorted data skip more? Look inside the Parquet footer.** Write *orders* twice as a single file with small row groups (`option("parquet.block.size", 1024 * 1024)`, `coalesce(1)`): once as it is and once sorted by *amount*. With `pyarrow.parquet.ParquetFile(file).metadata`, count the row groups whose largest *amount* (column statistics, `max`) is at most 990, because a filter `amount > 990` could skip them.

Put the numbers in *unsorted_skip*, *sorted_skip* and the number of row groups in *n_groups*.

In [ ]:
check_true("4.3", unsorted_skip == 0 and sorted_skip > 0.8 * n_groups, "unsorted: no row group can be skipped; sorted: almost all can")

## B. Layout: partition folders, small files, splittable files

**4.4 How do partition folders help?** Write *orders* with `partitionBy("order_date")` into `layout/by_date`. How many `order_date=...` folders are there (*n_folders*)? Then read it back and filter one day (`2024-02-14`): copy the *PartitionFilters* of the scan into *partition_filter* (a string).

In [ ]:
check("4.4", n_folders)
check_true("4.4b", "2024-02-14" in partition_filter, "the scan should list the date in PartitionFilters")

**4.5 Create the small file problem, then fix it.** Write *orders* partitioned by *order_date* twice:
* `layout/small`: after `repartition(50)`
* `layout/fixed`: after `repartition("order_date")`

Count the Parquet files of each (*files_small*, *files_fixed*, files are in `layout/<name>/order_date=.../`) and the read partitions of each (*parts_small*, *parts_fixed*, from `rdd.getNumPartitions()`).

In [ ]:
check_true("4.5", files_small > 1000 and files_fixed == 91 and parts_small > 10 * parts_fixed, "small: thousands of files and many read tasks; fixed: one file per day")

**4.6 Which files can be split into several read tasks?** Write *orders* as one CSV file (`coalesce(1)`, with header), once plain (`csv_none`) and once gzip-compressed (`option("compression", "gzip")`, `csv_gzip`). How many read partitions does each have (*plain_parts*, *gzip_parts*)? Then set `spark.sql.files.maxPartitionBytes` to `16m` and count the plain one again (*plain_16*). Reset the setting afterwards.

In [ ]:
check("4.6", [plain_parts, gzip_parts, plain_16])

**4.7 What does schema evolution look like in Parquet?** Write the 1,000 orders with `order_id <= 1000` to `evo/batch=1`, and the same orders plus a new column *discount* (value `0.1`) to `evo/batch=2`. Read the folder `evo` with `option("mergeSchema", "true")` into *merged*. Put the column names in *merged_cols* and the number of batch-1 rows where *discount* is null in *batch1_nulls*.

In [ ]:
check("4.7", [merged_cols, batch1_nulls])

**4.8 Which filters can not be pushed down, and how do you fix them?** On *parquet_df*, copy the *PushedFilters* (sorted lists, `[]` if none) of these four filters into a dict *pushed_by*:
* `lower_status`: `F.lower("status") == "paid"`
* `cast_amount`: `F.col("amount").cast("string") == "5.99"`
* `raw_status`: `F.col("status") == "paid"`
* `raw_amount`: `F.col("amount") == 5.99`

In [ ]:
check("4.8", pushed_by)

## C. Coding practice: dates and complex types

Four problems with date functions, a date spine, arrays and JSON. Write them from scratch. (Tip: collect dates or text, not timestamps, so your answer does not depend on your time zone.)

**4.9 Weekly revenue.** Group the *paid* orders by week (`date_trunc("week", "order_ts")`, weeks start on Monday; cast the week to a date) and sum the *amount*. Put `[number of weeks, the week with the highest revenue as text YYYY-MM-DD, its revenue rounded to 2 decimals]` in *result*.

In [ ]:
check("4.9", result)

**4.10 Fill the missing days.** Count the *returned* orders paid by *wallet* with an *amount* over 990 for every day from `2024-01-01` to `2024-03-31` (91 days), including the days with no such order (count 0). Build the list of days with `sequence` and `explode`, then left join. Put `[days_without, max_in_a_day]` in *result*: the number of days with a count of 0 and the largest daily count.

In [ ]:
check("4.10", result)

**4.11 Which product categories did each customer return?** For the customers 2 to 6, collect the distinct *category* of their *returned* orders into a sorted array. *cats* = list of `(customer_id, [categories...])` sorted by *customer_id*.

In [ ]:
check("4.11", cats)

**4.12 Parse JSON lines.** *raw* has one JSON text per row (one of them is not JSON at all). Parse the column *line* with `from_json` and the schema `order_id long, items array<struct<sku:string,qty:int>>`. How many rows could not be parsed (*n_bad*), and what is the total *qty* over all items of the good rows (*total_qty*)? Put `[total_qty, n_bad]` in *result*.

In [ ]:
raw = spark.createDataFrame([
    ('{"order_id": 1, "items": [{"sku": "a", "qty": 2}, {"sku": "b", "qty": 1}]}',),
    ('{"order_id": 2, "items": [{"sku": "a", "qty": 5}]}',),
    ('{"order_id": 3, "items": []}',),
    ('not json at all',),
], "line string")   # given


In [ ]:
check("4.12", result)

---
Done? In the Theory part of Day 4, answer the red checklist topics aloud, with the sizes, plans and file counts from this lab as evidence.